In [3]:
from dotenv import load_dotenv
load_dotenv()  # 在 os.getenv 之前调用
from langchain_openai import ChatOpenAI
import os

os.environ['OPENAI_API_KEY'] = os.getenv("DASHSCOPE_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("DASHSCOPE_BASE_URL")

chatLLM = ChatOpenAI(
  model="qwen-plus",
)


In [6]:
from langchain.prompts import PromptTemplate


prompt = PromptTemplate(
  template="请评价{topic}的优缺点，包括{aspects1}和{aspects2}。"
)
template = prompt.format(topic="电脑", aspects1="性能", aspects2="电池")

chatLLM.invoke(template)

AIMessage(content='您的问题中提到“请评价电脑的优缺点，包括性能和电池”，但未明确具体是哪款电脑（如品牌、型号、类型：笔记本/台式机/二合一，或是否指某类电脑如轻薄本、游戏本、MacBook、Chromebook等）。由于不同电脑差异极大，泛泛而谈“电脑”缺乏实际参考价值。为提供真正有用的信息，我为您分层次说明：\n\n✅ **合理理解与补充说明：**  \n“电脑”本身是一个宽泛类别，其优缺点高度依赖具体形态与定位：\n\n| 类型         | 性能特点（典型）                          | 电池续航（典型）               | 主要优势                          | 主要局限                          |\n|--------------|------------------------------------------|------------------------------|-----------------------------------|-----------------------------------|\n| **轻薄笔记本**<br>（如MacBook Air M3 / ThinkPad X1 Carbon） | 中高负载流畅（办公/编程/轻度创作）；能效比极佳；多核性能强于同功耗x86 | 12–18小时（网页/文档）；实测视频播放约10–14h | 极致便携、静音无风扇、长续航、系统稳定 | 显卡弱（不适用3D渲染/大型游戏）、扩展性差、接口少、维修成本高 |\n| **高性能笔记本**<br>（如ROG枪神、MacBook Pro 16" M3 Max） | 桌面级性能（多核渲染/编译/游戏），但持续高负载会降频、发热明显 | 通常5–9小时（日常使用）；满载时<2小时 | 移动工作站级生产力，兼顾游戏与专业创作 | 重（2.2–2.8kg）、厚、电池衰减快、噪音大、续航短、价格高 |\n| **台式电脑** | 性能天花板高（可配顶级CPU/GPU/内存/散热），升级灵活，长期稳定 | ❌ 无内置电池（依赖UPS应急供电） | 性价比高、散热优秀、扩展性强、静音潜力大 | 完全不移动、占用空间大、需外接电源、无续航能力 

In [10]:
from langchain_core.prompts import ChatPromptTemplate
#参数类型这里使用的是tuple构成的list
prompt_template = ChatPromptTemplate([
# 字符串 role + 字符串 content
("system", "你是一个AI开发工程师. 你的名字是 {name}."),
("human", "你能开发哪些AI应用?"),
("ai", "我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等."),
("human", "{user_input}")
])
#调用format()方法，返回字符串
prompt = prompt_template.invoke(input={"name":"小谷AI","user_input":"你能帮我做什么?"})
print(type(prompt))
print(prompt)

<class 'langchain_core.prompt_values.ChatPromptValue'>
messages=[SystemMessage(content='你是一个AI开发工程师. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='你能开发哪些AI应用?', additional_kwargs={}, response_metadata={}), AIMessage(content='我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.', additional_kwargs={}, response_metadata={}), HumanMessage(content='你能帮我做什么?', additional_kwargs={}, response_metadata={})]


In [11]:
from langchain.prompts.chat import ChatPromptTemplate
######1、提供提示词#########
chat_prompt = ChatPromptTemplate.from_messages([
("system", "你是一个数学家，你可以计算任何算式"),
("human", "我的问题：{question}"),
])
# 输入提示
messages = chat_prompt.format_messages(question="我今年18岁，我的舅舅今年38岁，我的爷爷今年72岁，我和舅舅一共多少岁了？")

######3、结合提示词，调用大模型#########
# 得到模型的输出
output = chatLLM.invoke(messages)
print(output)


content='我们来计算你和舅舅的年龄总和：\n\n- 你今年：18岁  \n- 舅舅今年：38岁  \n\n所以，你们一共：  \n18 + 38 = **56岁**\n\n✅ 答案：**56岁**。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 53, 'total_tokens': 114, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_name': 'qwen-plus', 'system_fingerprint': None, 'id': 'chatcmpl-93b9a533-43e1-979a-937d-228e78ff631f', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None} id='run--019d0040-7167-78c2-9c2f-93e31629169d-0' usage_metadata={'input_tokens': 53, 'output_tokens': 61, 'total_tokens': 114, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


In [ ]:
#1.导入相关包
from langchain.prompts import SystemMessagePromptTemplate
from langchain_core.messages import SystemMessage
from langchain_core.prompts import (ChatPromptTemplate, HumanMessagePromptTemplate,
MessagesPlaceholder)
# 2.定义消息模板
prompt = ChatPromptTemplate.from_messages([
SystemMessagePromptTemplate.from_template("你是{role}"),
MessagesPlaceholder(variable_name="intermediate_steps"),
HumanMessagePromptTemplate.from_template("{query}")
])
# 3.定义消息对象（运行时填充中间步骤的结果）
intermediate = [
SystemMessage(name="search", content="北京: 晴, 25°C")
]
# 4.格式化聊天消息提示词模版
prompt.format_messages(
role="天气预报员",
intermediate_steps=intermediate,
query="北京天气怎么样？"
)


In [12]:
from langchain.prompts import PromptTemplate
from langchain.prompts.few_shot import FewShotPromptTemplate
#1、创建示例集合
examples = [
{"input": "北京天气怎么样", "output": "北京市"},
{"input": "南京下雨吗", "output": "南京市"},
{"input": "武汉热吗", "output": "武汉市"}
]
#2、创建PromptTemplate实例
example_prompt = PromptTemplate.from_template(
template="Input: {input}\nOutput: {output}"
)
#3、创建FewShotPromptTemplate实例
prompt = FewShotPromptTemplate(
examples=examples,
example_prompt=example_prompt,
suffix="Input: {input}\nOutput:", # 要放在示例后面的提示模板字符串。
input_variables=["input"] # 传入的变量
)
#4、调用
prompt = prompt.invoke({"input":"长沙多少度"})
print("===Prompt===")
print(prompt)


===Prompt===
text='Input: 北京天气怎么样\nOutput: 北京市\n\nInput: 南京下雨吗\nOutput: 南京市\n\nInput: 武汉热吗\nOutput: 武汉市\n\nInput: 长沙多少度\nOutput:'
